# SQL Workforce Analytics

This notebook analyzes the synthetic Workforce Intelligence Platform data stored in PostgreSQL.

The analysis focuses on:

- Active workforce headcount
- Department distribution
- Geographic distribution
- Job-family distribution
- Annual turnover
- Recruiting funnel performance
- Requisition duration
- Compensation
- Performance reviews
- Employee training

The notebook queries PostgreSQL directly rather than reading the raw CSV files.

In [1]:
from pathlib import Path
import os

import pandas as pd
import psycopg2
from dotenv import load_dotenv


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


ENV_PATH = PROJECT_ROOT / ".env"


load_dotenv(
    ENV_PATH
)


database_config = {
    "host": os.getenv(
        "DB_HOST"
    ),
    "port": os.getenv(
        "DB_PORT"
    ),
    "dbname": os.getenv(
        "DB_NAME"
    ),
    "user": os.getenv(
        "DB_USER"
    ),
    "password": os.getenv(
        "DB_PASSWORD"
    ),
}


connection = psycopg2.connect(
    **database_config
)


print(
    "Connected to PostgreSQL."
)

Connected to PostgreSQL.


In [2]:
def run_query(
    query: str,
) -> pd.DataFrame:
    """Run a SQL query and return a DataFrame."""

    with connection.cursor() as cursor:

        cursor.execute(
            query
        )

        columns = [
            description[0]
            for description
            in cursor.description
        ]

        rows = (
            cursor.fetchall()
        )

    return pd.DataFrame(
        rows,
        columns=columns,
    )

## 1. Active headcount by department

In [3]:
department_headcount = run_query(
    """
    SELECT
        d.department_name,

        COUNT(*) FILTER (
            WHERE
                e.employment_status = 'Active'
        ) AS active_headcount,

        COUNT(*) FILTER (
            WHERE
                e.employment_status = 'Terminated'
        ) AS terminated_employee_count,

        COUNT(*) AS total_employee_records

    FROM employees AS e

    JOIN departments AS d
        ON e.department_id
           = d.department_id

    GROUP BY
        d.department_name

    ORDER BY
        active_headcount DESC;
    """
)

department_headcount

,department_name,active_headcount,terminated_employee_count,total_employee_records
0,Manufacturing,1971,528,2499
1,Engineering,1695,305,2000
2,Supply Chain,1015,185,1200
3,Sales,847,153,1000
4,Information Technology,836,164,1000
5,Finance,678,122,800
6,Customer Support,645,155,800
7,Human Resources,600,101,701


In [4]:
active_employee_count = run_query(
    """
    SELECT
        COUNT(*) AS active_employee_count
    FROM employees
    WHERE
        employment_status = 'Active';
    """
)

active_employee_count

,active_employee_count
0,8287


In [5]:
assert (
    department_headcount[
        "active_headcount"
    ].sum()
    ==
    active_employee_count.loc[
        0,
        "active_employee_count",
    ]
)

print(
    "Department headcount check passed."
)

Department headcount check passed.


## 2. 2025 department turnover

In [6]:
turnover_2025 = run_query(
    """
    WITH department_headcount AS (
        SELECT
            d.department_name,

            COUNT(*) FILTER (
                WHERE
                    e.hire_date <= DATE '2025-01-01'
                    AND (
                        e.termination_date IS NULL
                        OR e.termination_date > DATE '2025-01-01'
                    )
            ) AS starting_headcount,

            COUNT(*) FILTER (
                WHERE
                    e.hire_date <= DATE '2025-12-31'
                    AND (
                        e.termination_date IS NULL
                        OR e.termination_date > DATE '2025-12-31'
                    )
            ) AS ending_headcount,

            COUNT(*) FILTER (
                WHERE
                    e.termination_date
                    BETWEEN DATE '2025-01-01'
                    AND DATE '2025-12-31'
            ) AS terminations_2025

        FROM employees AS e

        JOIN departments AS d
            ON e.department_id
               = d.department_id

        GROUP BY
            d.department_name
    )

    SELECT
        department_name,
        starting_headcount,
        ending_headcount,
        terminations_2025,

        ROUND(
            (
                starting_headcount
                + ending_headcount
            ) / 2.0,
            2
        ) AS average_headcount,

        ROUND(
            100.0
            * terminations_2025
            / NULLIF(
                (
                    starting_headcount
                    + ending_headcount
                ) / 2.0,
                0
            ),
            2
        ) AS turnover_rate_percent

    FROM department_headcount

    ORDER BY
        turnover_rate_percent DESC;
    """
)

turnover_2025

,department_name,starting_headcount,ending_headcount,terminations_2025,average_headcount,turnover_rate_percent
0,Manufacturing,1672,1921,169,1796.50,9.41
1,Customer Support,545,621,53,583.00,9.09
2,Information Technology,702,808,56,755.00,7.42
3,Supply Chain,839,964,66,901.50,7.32
4,Finance,537,633,41,585.00,7.01
5,Engineering,1398,1613,104,1505.50,6.91
6,Sales,663,782,48,722.50,6.64
7,Human Resources,487,565,33,526.00,6.27


## 3. Recruiting funnel by source

In [7]:
recruiting_funnel = run_query(
    """
    SELECT
        c.application_source,

        COUNT(
            DISTINCT c.candidate_id
        ) AS candidate_count,

        COUNT(
            a.application_id
        ) AS application_count,

        COUNT(
            a.application_id
        ) FILTER (
            WHERE
                a.interview_score IS NOT NULL
        ) AS interviewed_applications,

        COUNT(
            a.application_id
        ) FILTER (
            WHERE
                a.offer_date IS NOT NULL
        ) AS offers,

        COUNT(
            a.application_id
        ) FILTER (
            WHERE
                a.application_status = 'Hired'
        ) AS hires,

        ROUND(
            100.0
            * COUNT(
                a.application_id
            ) FILTER (
                WHERE
                    a.application_status = 'Hired'
            )
            / NULLIF(
                COUNT(
                    a.application_id
                ),
                0
            ),
            2
        ) AS application_to_hire_rate_percent

    FROM candidates AS c

    JOIN applications AS a
        ON c.candidate_id
           = a.candidate_id

    GROUP BY
        c.application_source

    ORDER BY
        hires DESC;
    """
)

recruiting_funnel

,application_source,candidate_count,application_count,interviewed_applications,offers,hires,application_to_hire_rate_percent
0,LinkedIn,11093,14715,8591,3412,2416,16.42
1,Employee Referral,5263,6464,4284,2618,2269,35.10
2,Company Careers Page,8320,11006,6395,2692,1966,17.86
3,Indeed,6698,8933,5024,1874,1237,13.85
4,University Recruiting,2884,3747,2241,1058,811,21.64
5,Staffing Agency,2590,3503,2054,789,522,14.90
6,Professional Association,1716,2182,1357,643,508,23.28
7,Job Fair,1436,1886,1055,401,271,14.37


In [8]:
assert (
    recruiting_funnel[
        "hires"
    ].sum()
    == 10_000
)

print(
    "Recruiting hire-count check passed."
)

Recruiting hire-count check passed.


## 4. Current compensation by department

In [9]:
compensation_summary = run_query(
    """
    WITH latest_compensation AS (
        SELECT DISTINCT ON (
            employee_id
        )
            employee_id,
            base_salary,
            bonus_target,
            equity_value

        FROM compensation_history

        WHERE
            effective_date
            <= DATE '2026-06-30'

        ORDER BY
            employee_id,
            effective_date DESC,
            compensation_id DESC
    )

    SELECT
        d.department_name,

        COUNT(*) AS active_employee_count,

        ROUND(
            AVG(
                lc.base_salary
            ),
            2
        ) AS average_base_salary,

        ROUND(
            AVG(
                lc.bonus_target
            ),
            2
        ) AS average_bonus_target,

        ROUND(
            AVG(
                lc.equity_value
            ),
            2
        ) AS average_equity_value

    FROM employees AS e

    JOIN departments AS d
        ON e.department_id
           = d.department_id

    JOIN latest_compensation AS lc
        ON e.employee_id
           = lc.employee_id

    WHERE
        e.employment_status = 'Active'

    GROUP BY
        d.department_name

    ORDER BY
        average_base_salary DESC;
    """
)

compensation_summary

,department_name,active_employee_count,average_base_salary,average_bonus_target,average_equity_value
0,Engineering,1695,124785.43,12.10,12101.77
1,Information Technology,836,116282.66,11.05,8727.87
2,Supply Chain,1015,88479.80,9.74,6231.53
3,Finance,678,88342.04,9.28,5026.55
4,Manufacturing,1971,88006.09,8.81,7684.17
5,Human Resources,600,85305.83,9.29,4991.67
6,Sales,847,82563.05,9.63,6224.32
7,Customer Support,645,57878.14,3.59,752.71


## 5. Latest performance by department

In [10]:
performance_summary = run_query(
    """
    WITH latest_review AS (
        SELECT DISTINCT ON (
            employee_id
        )
            employee_id,
            performance_rating,
            goal_completion,
            promotion_recommended

        FROM performance_reviews

        ORDER BY
            employee_id,
            review_date DESC,
            review_id DESC
    )

    SELECT
        d.department_name,

        COUNT(
            lr.employee_id
        ) AS employees_with_reviews,

        ROUND(
            AVG(
                lr.performance_rating
            ),
            2
        ) AS average_performance_rating,

        ROUND(
            AVG(
                lr.goal_completion
            ),
            2
        ) AS average_goal_completion,

        ROUND(
            100.0
            * COUNT(*) FILTER (
                WHERE
                    lr.promotion_recommended = TRUE
            )
            / NULLIF(
                COUNT(
                    lr.employee_id
                ),
                0
            ),
            2
        ) AS promotion_recommendation_rate_percent

    FROM employees AS e

    JOIN departments AS d
        ON e.department_id
           = d.department_id

    JOIN latest_review AS lr
        ON e.employee_id
           = lr.employee_id

    WHERE
        e.employment_status = 'Active'

    GROUP BY
        d.department_name

    ORDER BY
        average_performance_rating DESC;
    """
)

performance_summary

,department_name,employees_with_reviews,average_performance_rating,average_goal_completion,promotion_recommendation_rate_percent
0,Engineering,1401,3.62,93.55,14.13
1,Finance,532,3.56,92.97,10.53
2,Human Resources,493,3.56,93.07,10.75
3,Sales,676,3.55,92.95,13.31
4,Supply Chain,835,3.53,92.68,9.82
5,Information Technology,687,3.53,92.35,9.02
6,Manufacturing,1608,3.44,91.18,9.27
7,Customer Support,519,3.38,89.88,8.09


## 6. Training activity by department

In [11]:
training_summary = run_query(
    """
    WITH training_by_employee AS (
        SELECT
            employee_id,

            COUNT(*) FILTER (
                WHERE
                    completion_status = 'Completed'
            ) AS completed_programs,

            COALESCE(
                SUM(
                    training_hours
                ) FILTER (
                    WHERE
                        completion_status = 'Completed'
                ),
                0
            ) AS completed_training_hours

        FROM training_records

        GROUP BY
            employee_id
    )

    SELECT
        d.department_name,

        COUNT(
            e.employee_id
        ) AS active_employee_count,

        ROUND(
            AVG(
                COALESCE(
                    t.completed_programs,
                    0
                )
            ),
            2
        ) AS average_completed_programs,

        ROUND(
            AVG(
                COALESCE(
                    t.completed_training_hours,
                    0
                )
            ),
            2
        ) AS average_completed_training_hours

    FROM employees AS e

    JOIN departments AS d
        ON e.department_id
           = d.department_id

    LEFT JOIN training_by_employee AS t
        ON e.employee_id
           = t.employee_id

    WHERE
        e.employment_status = 'Active'

    GROUP BY
        d.department_name

    ORDER BY
        average_completed_training_hours DESC;
    """
)

training_summary

,department_name,active_employee_count,average_completed_programs,average_completed_training_hours
0,Manufacturing,1971,3.22,41.23
1,Supply Chain,1015,3.21,40.92
2,Information Technology,836,2.42,34.77
3,Engineering,1695,2.39,34.26
4,Customer Support,645,2.20,22.84
5,Human Resources,600,1.41,16.63
6,Sales,847,1.37,15.43
7,Finance,678,1.34,15.15


## 7. Analytics validation

In [12]:
total_active = int(
    active_employee_count.loc[
        0,
        "active_employee_count",
    ]
)


analytics_checks = pd.Series(
    {
        "eight departments appear": (
            len(
                department_headcount
            )
            == 8
        ),

        "department active counts match total active employees": (
            int(
                department_headcount[
                    "active_headcount"
                ].sum()
            )
            == total_active
        ),

        "recruiting funnel contains all 10,000 hires": (
            int(
                recruiting_funnel[
                    "hires"
                ].sum()
            )
            == 10_000
        ),

        "compensation analysis contains departments": (
            len(
                compensation_summary
            )
            > 0
        ),

        "performance analysis contains departments": (
            len(
                performance_summary
            )
            > 0
        ),

        "training analysis contains departments": (
            len(
                training_summary
            )
            > 0
        ),
    },
    name="passed",
)

analytics_checks

eight departments appear                                 True
department active counts match total active employees    True
recruiting funnel contains all 10,000 hires              True
compensation analysis contains departments               True
performance analysis contains departments                True
training analysis contains departments                   True
Name: passed, dtype: bool

In [13]:
if analytics_checks.all():
    print(
        "All SQL analytics validation "
        "checks passed."
    )
else:
    print(
        "One or more SQL analytics "
        "checks failed."
    )

All SQL analytics validation checks passed.


In [14]:
connection.close()

print(
    "Database connection closed."
)

Database connection closed.


## 8. Conclusions

The SQL workforce analytics layer successfully combines data from the platform's workforce systems.

### Workforce analysis

- Active headcount can be analyzed by department, location, and job family.
- Historical termination data can be used to estimate annual department turnover.
- Department-level workforce comparisons are available directly from PostgreSQL.

### Recruiting analysis

- Candidate sources can be compared across applications, interviews, offers, and hires.
- Filled, cancelled, and open requisitions can be analyzed by recruiting duration.
- The recruiting tables connect candidates, requisitions, applications, and employees.

### Employee development analysis

- Current compensation is derived from each employee's latest compensation record.
- Current performance analysis uses each employee's latest performance review.
- Training records can be aggregated to the employee and department levels.

### Next steps

The SQL analytics layer provides a foundation for building an employee-level analytical dataset for retention analysis and machine-learning modeling.

### Current limitations

- Historical department transfers are not fully reconstructed when calculating department-level historical turnover.
- The project uses synthetic data and simulated business processes.
- Current compensation and performance are based on the latest available records as of the project analysis date.